# Análise: Público dos contatos WhatsApp e motivos de não fechamento

Dados da aba **whatsapp_leads** (`Consultório Psicologia.xlsx`).

**Objetivos:**
1. Entender **quem está chegando**: origem (estado), perfil (sexo), volume no tempo.
2. Entender **por que não fecharam**: distribuição de motivos e cruzamentos (motivo × estado, motivo × tempo).

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", palette="husl")

In [ ]:
df = pd.read_excel("src/Consultório Psicologia.xlsx", sheet_name="whatsapp_leads")
df["data_contato"] = pd.to_datetime(df["data_contato"], errors="coerce")
df.head()

---
## 1. Entender o público que está chegando

### 1.1 Contatos por Estado (UF)
Mostra **de onde** vêm os contatos — priorizar divulgação ou ajuste de mensagem por região.

In [ ]:
por_estado = df["estado"].value_counts().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
por_estado.plot(kind="barh", ax=ax)
ax.set_title("Contatos por Estado (origem do público)")
ax.set_xlabel("Quantidade")
plt.tight_layout()
plt.show()

### 1.2 Contatos por Sexo
Perfil demográfico — com mais dados, ajuda a direcionar comunicação.

In [ ]:
por_sexo = df["sexo"].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
por_sexo.plot(kind="bar", ax=ax, color=["#2ecc71", "#3498db"])
ax.set_title("Contatos por Sexo")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

### 1.3 Contatos ao longo do tempo
Volume por data — identifica picos e períodos com mais demanda.

In [ ]:
por_data = df.groupby(df["data_contato"].dt.date).size()
fig, ax = plt.subplots(figsize=(10, 4))
por_data.plot(kind="bar", ax=ax)
ax.set_title("Contatos por dia")
ax.set_xlabel("Data")
ax.set_ylabel("Quantidade")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 1.4 Contatos por dia da semana
Em quais dias as pessoas mais entram em contato — útil para disponibilidade e campanhas.

In [ ]:
dias = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
df["dia_semana_idx"] = df["data_contato"].dt.dayofweek
por_dia = df["dia_semana_idx"].value_counts().sort_index().reindex(range(7), fill_value=0)
por_dia.index = dias
fig, ax = plt.subplots(figsize=(8, 4))
por_dia.plot(kind="bar", ax=ax)
ax.set_title("Contatos por dia da semana")
ax.set_xlabel("Dia")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 2. Por que não fecharam

### 2.1 Distribuição dos motivos
Principal visão: **qual motivo mais explica** os não fechamentos (Não retornou, Preço, Público Errado, etc.).

In [ ]:
por_motivo = df["motivo"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
por_motivo.plot(kind="bar", ax=axes[0])
axes[0].set_title("Contatos por motivo (não fechou)")
axes[0].tick_params(axis="x", rotation=45)
por_motivo.plot(kind="pie", ax=axes[1], autopct="%1.0f%%", startangle=90)
axes[1].set_title("Proporção por motivo")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

### 2.2 Motivo × Estado
Em **quais estados** cada motivo mais aparece — ex.: Preço em MG, Público Errado em SP.

In [ ]:
cross = pd.crosstab(df["estado"], df["motivo"])
fig, ax = plt.subplots(figsize=(10, 5))
cross.plot(kind="bar", ax=ax, stacked=False)
ax.set_title("Motivo de não fechamento por Estado")
ax.set_xlabel("Estado")
ax.legend(title="Motivo", bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 2.3 Motivo ao longo do tempo
Se os motivos mudam com o tempo — ex.: mais "Preço" em um período após reajuste.

In [ ]:
por_data_motivo = pd.crosstab(df["data_contato"].dt.date, df["motivo"])
por_data_motivo.plot(kind="bar", stacked=True, figsize=(10, 4))
plt.title("Motivos de não fechamento ao longo do tempo")
plt.xlabel("Data")
plt.ylabel("Quantidade")
plt.legend(title="Motivo", bbox_to_anchor=(1.02, 1))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 3. Análise semanal e mensal

Visão focada na **semana atual** (últimos 7 dias) e no **mês atual**, para acompanhamento operacional e comparação rápida.

In [ ]:
# Referência: hoje; semana = últimos 7 dias; mês = desde dia 1 do mês atual
hoje = pd.Timestamp.today().normalize()
inicio_semana = hoje - pd.Timedelta(days=6)  # 7 dias incluindo hoje
inicio_mes = hoje.replace(day=1)

df_semana = df[df["data_contato"].dt.normalize() >= inicio_semana].copy()
df_mes = df[df["data_contato"].dt.normalize() >= inicio_mes].copy()

print(f"Semana: {inicio_semana.date()} a {hoje.date()} → {len(df_semana)} contatos")
print(f"Mês: desde {inicio_mes.date()} → {len(df_mes)} contatos")

### 3.1 Volume: semana vs mês
Métricas rápidas para acompanhamento (alinhado ao dashboard do app).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, val, label in zip(
    axes,
    [len(df_semana), len(df_mes), len(df)],
    ["Contatos na semana", "Contatos no mês", "Total (histórico)"],
):
    ax.text(0.5, 0.6, str(val), ha="center", fontsize=28)
    ax.text(0.5, 0.25, label, ha="center", fontsize=11)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
plt.suptitle("Volume de contatos", y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Motivos: semana vs mês (lado a lado)
Comparar **por que não fecharam** na semana atual e no mês — ver se o padrão recente difere do mês todo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, frame, tit in zip(axes, [df_semana, df_mes], ["Últimos 7 dias (semana)", "Mês atual"]):
    if frame.empty:
        ax.text(0.5, 0.5, "Sem dados", ha="center", va="center")
    else:
        frame["motivo"].value_counts().plot(kind="bar", ax=ax)
    ax.set_title(tit)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### 3.3 Público por Estado: semana vs mês
De onde vêm os contatos na janela recente (semana) vs no mês — priorizar regiões em campanhas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, frame, tit in zip(axes, [df_semana, df_mes], ["Estados – última semana", "Estados – mês atual"]):
    if frame.empty:
        ax.text(0.5, 0.5, "Sem dados", ha="center", va="center")
    else:
        frame["estado"].value_counts().sort_values(ascending=True).plot(kind="barh", ax=ax)
    ax.set_title(tit)
    ax.set_xlabel("Quantidade")
plt.tight_layout()
plt.show()

### 3.4 Evolução mês a mês
Contatos por mês (série temporal) — tendência de volume e comparação entre meses.

In [ ]:
df["mes"] = df["data_contato"].dt.to_period("M")
por_mes = df.groupby("mes").size()
fig, ax = plt.subplots(figsize=(10, 4))
por_mes.plot(kind="bar", ax=ax)
ax.set_title("Contatos por mês (evolução)")
ax.set_xlabel("Mês")
ax.set_ylabel("Quantidade")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 3.5 Motivos por mês (empilhado)
Ver como a **mistura de motivos** muda mês a mês (ex.: mais Preço em um mês específico).

In [ ]:
cross_mes = pd.crosstab(df["mes"].astype(str), df["motivo"])
if not cross_mes.empty:
    cross_mes.plot(kind="bar", stacked=True, figsize=(10, 4))
    plt.title("Motivos de não fechamento por mês")
    plt.xlabel("Mês")
    plt.ylabel("Quantidade")
    plt.legend(title="Motivo", bbox_to_anchor=(1.02, 1))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Sem dados suficientes para motivos por mês.")

---
## Resumo das sugestões de gráficos

| Objetivo | Gráfico | O que responde |
|----------|--------|----------------|
| **Público** | Contatos por Estado | De onde vêm os contatos |
| **Público** | Contatos por Sexo | Perfil demográfico |
| **Público** | Contatos por data | Volume e picos no tempo |
| **Público** | Contatos por dia da semana | Quando mais procuram |
| **Não fechou** | Barras/Pizza por motivo | Principal razão (Preço, Não retornou, Público Errado) |
| **Não fechou** | Motivo × Estado | Onde cada motivo mais aparece |
| **Não fechou** | Motivo × tempo | Tendência dos motivos ao longo do tempo |
| **Semanal/Mensal** | Volume semana vs mês vs total | Métricas rápidas de acompanhamento |
| **Semanal/Mensal** | Motivos: semana vs mês (lado a lado) | Padrão recente vs mês todo |
| **Semanal/Mensal** | Estado: semana vs mês | Origem do público na janela recente |
| **Semanal/Mensal** | Evolução mês a mês | Tendência de volume entre meses |
| **Semanal/Mensal** | Motivos por mês (empilhado) | Como a mistura de motivos muda no tempo |